# 핸즈온 04 — Function Calling으로 ITS 관제 에이전트 만들기

**소요 시간**: 80~90분
**학습 목표**:
1. **Function Calling**의 동작 원리를 이해한다
2. ITS 운영시스템의 함수들을 Gemini가 호출할 수 있는 tool로 등록한다
3. **CCTV 분석 → DB 조회 → 알람 발송**까지 자동화하는 멀티스텝 에이전트를 구현한다
4. Function Calling + Built-in Tools(Google Search) 조합 패턴을 익힌다

> **운영 환경 적용 시 주의**: 본 핸즈온의 함수들은 모두 **모킹(mocking)**되어 있습니다. 실제 ITS 시스템 연동 시에는 발주처의 보안 정책(망분리, 인증, 감사로그)을 반드시 준수해야 합니다. 운영 배포 시 **Vertex AI(asia-northeast3)** 사용을 권장합니다.

## 4-1. 환경 셋업

In [ ]:
!pip install -q -U google-genai pydantic

In [ ]:
import os, json, time, random
from datetime import datetime
from typing import Literal
from google import genai
from google.genai import types
from pydantic import BaseModel

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("✅ Client ready")

## 4-2. Function Calling 동작 원리

```
사용자 질의
    ↓
모델이 어떤 함수를 호출할지 결정
    ↓
함수 호출 정보(이름 + 인자)를 클라이언트에 반환
    ↓
클라이언트가 실제 함수 실행
    ↓
결과를 다시 모델에 전달
    ↓
모델이 최종 응답 생성
```

**모델은 함수를 직접 실행하지 않습니다**. 어떤 함수를 호출할지 알려주고, 실행은 우리가 합니다. 이게 보안적으로 매우 중요합니다.

## 4-3. 워밍업 — 가장 단순한 Function Calling

날씨 조회 함수를 모킹하고, 모델이 이걸 어떻게 호출하는지 봅니다.

In [ ]:
def get_weather(city: str, date: str = "today") -> dict:
    """특정 도시의 날씨를 조회합니다."""
    # 모킹된 응답
    mock_data = {
        "서울": {"temperature": 18, "condition": "맑음", "rain_mm": 0},
        "수원": {"temperature": 17, "condition": "흐림", "rain_mm": 0.2},
        "인천": {"temperature": 16, "condition": "비", "rain_mm": 5.3},
    }
    return mock_data.get(city, {"error": "데이터 없음"})


# Function declaration — 모델에게 알려줄 함수 명세
weather_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="get_weather",
        description="특정 도시의 날씨 정보를 조회합니다",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "city": types.Schema(type=types.Type.STRING, description="도시명 (예: 서울)"),
                "date": types.Schema(type=types.Type.STRING, description="조회 날짜 (today/tomorrow)"),
            },
            required=["city"],
        ),
    )
])

# 사용자 질의
resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="수원 날씨 어때?",
    config=types.GenerateContentConfig(tools=[weather_tool]),
)

# 응답 안에 함수 호출이 들어있는지 확인
fc = resp.candidates[0].content.parts[0].function_call
if fc:
    print(f"📞 모델이 호출하려는 함수: {fc.name}")
    print(f"   인자: {dict(fc.args)}")
else:
    print(f"💬 일반 응답: {resp.text}")

### 함수 실행 → 결과를 모델에 전달 → 최종 응답

이게 1턴짜리 function calling의 완전한 흐름입니다.

In [ ]:
from google.genai.types import FunctionResponse, Part, Content

# 사용자 질의
user_query = "수원 날씨 어때? 우산 챙겨야 해?"

# 1라운드: 모델에게 질문 + tool 제공
chat_history = [Content(role="user", parts=[Part(text=user_query)])]
resp1 = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=chat_history,
    config=types.GenerateContentConfig(tools=[weather_tool]),
)

# 모델 응답 (함수 호출)
model_msg = resp1.candidates[0].content
chat_history.append(model_msg)

fc = model_msg.parts[0].function_call
print(f"🤖 모델: {fc.name}({dict(fc.args)}) 호출 요청")

# 우리가 실제 함수 실행
result = get_weather(**dict(fc.args))
print(f"⚙️  실행 결과: {result}\n")

# 2라운드: 함수 결과를 모델에 전달
chat_history.append(Content(
    role="user",
    parts=[Part(function_response=FunctionResponse(name=fc.name, response=result))],
))
resp2 = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=chat_history,
    config=types.GenerateContentConfig(tools=[weather_tool]),
)
print(f"🤖 최종 응답:\n{resp2.text}")

## 4-4. ITS 미니 에이전트 — 도구 정의

이제 본격적으로 ITS 관제 시나리오를 모킹합니다.

### 가상의 ITS 시스템 함수 4종
1. `get_traffic_status(road_id)` — 특정 도로 구간의 실시간 소통 상태
2. `query_recent_incidents(area, hours)` — 최근 N시간 사고 이력 조회
3. `dispatch_patrol_unit(road_id, reason)` — 순찰차 출동 요청
4. `send_alert_to_dms(road_id, message)` — 도로전광표지(DMS)에 메시지 송출

In [ ]:
# === 모킹된 ITS 시스템 함수들 ===

# 가상 도로 데이터베이스
ROAD_DB = {
    "GH-0001": {"name": "경부고속도로 서울TG~판교IC 상행", "length_km": 12.5},
    "GH-0002": {"name": "경부고속도로 서울TG~판교IC 하행", "length_km": 12.5},
    "GH-0103": {"name": "영동고속도로 안산JC~군포IC", "length_km": 8.3},
    "URB-0501": {"name": "외곽순환도로 시흥IC~광명IC", "length_km": 15.2},
}

INCIDENTS = [
    {"id": "INC-26042301", "road_id": "GH-0103", "time": "2026-04-23 14:32",
     "type": "추돌사고", "lanes_blocked": 2, "status": "진행중"},
    {"id": "INC-26042205", "road_id": "GH-0001", "time": "2026-04-22 17:15",
     "type": "낙하물", "lanes_blocked": 1, "status": "처리완료"},
    {"id": "INC-26042102", "road_id": "URB-0501", "time": "2026-04-21 09:08",
     "type": "단독사고", "lanes_blocked": 1, "status": "처리완료"},
]

DISPATCHED = []
DMS_QUEUE = []


def get_traffic_status(road_id: str) -> dict:
    """특정 도로 구간의 실시간 소통 상태 조회."""
    if road_id not in ROAD_DB:
        return {"error": f"존재하지 않는 도로 ID: {road_id}"}

    # 시연용 가짜 데이터 — 실제로는 VDS API 호출
    info = ROAD_DB[road_id]
    avg_speed = random.choice([25, 45, 65, 85])
    levels = {25: "정체", 45: "지체", 65: "서행", 85: "원활"}
    return {
        "road_id": road_id,
        "name": info["name"],
        "avg_speed_kph": avg_speed,
        "level": levels[avg_speed],
        "volume_per_5min": random.randint(50, 300),
        "queried_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }


def query_recent_incidents(area: str = "all", hours: int = 24) -> dict:
    """최근 N시간 내 발생한 사고 이력 조회."""
    # 시연용으로는 시간 필터링 생략
    return {
        "count": len(INCIDENTS),
        "incidents": INCIDENTS,
        "area": area,
        "window_hours": hours,
    }


def dispatch_patrol_unit(road_id: str, reason: str) -> dict:
    """순찰차 출동 요청."""
    if road_id not in ROAD_DB:
        return {"success": False, "error": "존재하지 않는 도로 ID"}
    unit_id = f"PT-{random.randint(100, 999)}"
    eta_min = random.randint(5, 25)
    DISPATCHED.append({
        "unit_id": unit_id, "road_id": road_id, "reason": reason,
        "eta_min": eta_min, "dispatched_at": datetime.now().isoformat(),
    })
    return {
        "success": True,
        "unit_id": unit_id,
        "road_id": road_id,
        "eta_min": eta_min,
        "message": f"{unit_id} 출동, 약 {eta_min}분 후 현장 도착 예정",
    }


def send_alert_to_dms(road_id: str, message: str) -> dict:
    """도로전광표지(DMS)에 알림 메시지 송출."""
    if len(message) > 30:
        return {"success": False, "error": "메시지는 30자 이하여야 합니다"}
    DMS_QUEUE.append({
        "road_id": road_id, "message": message,
        "queued_at": datetime.now().isoformat(),
    })
    return {"success": True, "road_id": road_id, "broadcast_message": message}


# 실제 호출 가능한 함수 매핑
FUNCTION_MAP = {
    "get_traffic_status": get_traffic_status,
    "query_recent_incidents": query_recent_incidents,
    "dispatch_patrol_unit": dispatch_patrol_unit,
    "send_alert_to_dms": send_alert_to_dms,
}

print("✅ 모킹된 ITS 함수 4종 정의 완료")

### Function Declarations — 모델에게 알려줄 명세

In [ ]:
its_tools = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="get_traffic_status",
        description="특정 도로 구간의 실시간 소통 상태를 조회합니다. 평균 속도, 통행량, 혼잡 레벨을 반환합니다.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "road_id": types.Schema(type=types.Type.STRING,
                    description="도로 구간 ID (예: GH-0001)"),
            },
            required=["road_id"],
        ),
    ),
    types.FunctionDeclaration(
        name="query_recent_incidents",
        description="최근 N시간 내 발생한 사고/이상 이벤트 이력을 조회합니다.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "area": types.Schema(type=types.Type.STRING,
                    description="조회 영역 (예: 'all', '경부선', '영동선')"),
                "hours": types.Schema(type=types.Type.INTEGER,
                    description="조회 시간 범위 (시간 단위, 기본 24)"),
            },
            required=[],
        ),
    ),
    types.FunctionDeclaration(
        name="dispatch_patrol_unit",
        description="해당 도로 구간으로 순찰차 출동을 요청합니다. 사고나 낙하물 등 현장 확인이 필요할 때 사용합니다.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "road_id": types.Schema(type=types.Type.STRING, description="도로 구간 ID"),
                "reason": types.Schema(type=types.Type.STRING, description="출동 사유"),
            },
            required=["road_id", "reason"],
        ),
    ),
    types.FunctionDeclaration(
        name="send_alert_to_dms",
        description="도로전광표지(DMS)에 운전자 안내 메시지를 송출합니다. 메시지는 30자 이하.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "road_id": types.Schema(type=types.Type.STRING, description="도로 구간 ID"),
                "message": types.Schema(type=types.Type.STRING,
                    description="송출 메시지 (한글 30자 이하)"),
            },
            required=["road_id", "message"],
        ),
    ),
])

print("✅ Tool declarations 등록 완료 (4개 함수)")

## 4-5. 멀티스텝 에이전트 루프

여러 함수를 순차적으로 호출해야 하는 시나리오를 처리합니다.

In [ ]:
SYSTEM_PROMPT = """당신은 한국도로공사 교통관제센터의 AI 보조 에이전트입니다.
운영자의 자연어 지시를 받아 다음 도구를 활용해 작업을 수행합니다:
- get_traffic_status: 실시간 소통 상태 조회
- query_recent_incidents: 사고 이력 조회
- dispatch_patrol_unit: 순찰차 출동 요청
- send_alert_to_dms: 도로전광표지 메시지 송출

규칙:
1. 사고 발생 또는 정체 심화가 의심되면 반드시 현황을 먼저 확인하세요.
2. 순찰차 출동 또는 DMS 메시지 송출 같은 행동은 충분한 근거가 있을 때만 실행합니다.
3. 작업 결과를 운영자에게 명확히 보고하세요."""


def run_agent(user_query, max_turns=8, verbose=True):
    """멀티스텝 function calling 에이전트."""
    history = [Content(role="user", parts=[Part(text=user_query)])]

    for turn in range(max_turns):
        resp = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=history,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                tools=[its_tools],
                thinking_config=types.ThinkingConfig(thinking_level="medium"),
            ),
        )
        msg = resp.candidates[0].content
        history.append(msg)

        # 함수 호출이 있는지 확인
        function_calls = [p.function_call for p in msg.parts if p.function_call]

        if not function_calls:
            # 함수 호출 없음 → 최종 답변
            text_parts = [p.text for p in msg.parts if p.text]
            final_text = "\n".join(text_parts)
            if verbose:
                print(f"\n🤖 [최종 응답]\n{final_text}")
            return final_text, history

        # 모든 함수 호출 실행
        function_responses = []
        for fc in function_calls:
            args = dict(fc.args) if fc.args else {}
            if verbose:
                print(f"\n📞 [Turn {turn+1}] {fc.name}({args})")
            if fc.name in FUNCTION_MAP:
                try:
                    result = FUNCTION_MAP[fc.name](**args)
                except Exception as e:
                    result = {"error": f"실행 실패: {e}"}
            else:
                result = {"error": f"알 수 없는 함수: {fc.name}"}
            if verbose:
                print(f"   → {json.dumps(result, ensure_ascii=False)[:200]}")
            function_responses.append(Part(
                function_response=FunctionResponse(name=fc.name, response=result)
            ))

        # 결과를 다시 모델에 전달
        history.append(Content(role="user", parts=function_responses))

    return "최대 턴 초과", history


print("✅ Agent loop 정의 완료")

## 4-6. 시나리오 1 — 단순 조회

In [ ]:
run_agent("경부고속도로 서울TG에서 판교IC 상행 지금 어때?")

## 4-7. 시나리오 2 — 사고 인지 + 후속 조치 자동화

자연어 한 줄로 **여러 함수 호출 + 의사결정 + 액션**을 연쇄 실행하게 합니다.

In [ ]:
run_agent("""GH-0103 영동선에서 사고 신고가 들어왔어. 다음 작업 처리해줘:
1. 현재 영동선 GH-0103 소통 상태 확인
2. 최근 24시간 사고 이력 확인
3. 진행중인 사고가 있으면 순찰차 출동 요청
4. 후방 차량을 위한 DMS 메시지 송출 (30자 이하)""")

> **관찰 포인트**
>
> 1. 모델이 4개 함수를 어떤 순서로 호출했는가?
> 2. 각 함수 호출 사이에 추론 단계가 있었는가?
> 3. DMS 메시지가 30자 제약을 지켰는가?
> 4. 의도하지 않은 함수를 호출했는가?

## 4-8. 시나리오 3 — 운영 안전성 확인

위험 시나리오: 모델이 운영자 의도와 다르게 함수를 호출하지 않는지 확인합니다.

In [ ]:
run_agent("경부선 어디 한 군데 골라서 그냥 순찰차 보내봐")

> 위 응답에서 모델이:
> - 무작정 함수를 호출하는가, 아니면 명확한 사유를 묻는가?
> - **운영 시스템에서는 이런 안전장치가 매우 중요**합니다. 시스템 프롬프트의 "충분한 근거가 있을 때만 실행" 규칙이 작동하는지 보세요.

## 4-9. 결과 확인 — 호출된 액션 로그

In [ ]:
print("=" * 60)
print("📋 출동 요청 이력")
print("=" * 60)
for d in DISPATCHED:
    print(f"  {d['unit_id']} → {d['road_id']} ({d['reason']}) ETA {d['eta_min']}분")

print("\n" + "=" * 60)
print("📋 DMS 송출 이력")
print("=" * 60)
for m in DMS_QUEUE:
    print(f"  [{m['road_id']}] {m['message']}")

## 4-10. Built-in Tool 결합 — Google Search Grounding

ITS 운영자는 사내 시스템뿐 아니라 외부 정보도 필요합니다. 예: 기상청 특보, 뉴스 사고 속보. Google Search grounding을 함께 쓰면 됩니다.

> **주의**: Gemini 3에서 search grounding은 쿼리당 과금됩니다 ($14/1k queries). 강의 데모 1~2회는 거의 무료지만 운영 시 주의.

In [ ]:
# Built-in Search 도구
search_tool = types.Tool(google_search=types.GoogleSearch())

resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="오늘 수원 지역 강수 예보 알려줘. 도로 운영에 영향 있을 정도인지 판단해줘.",
    config=types.GenerateContentConfig(
        tools=[search_tool],
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)
print(resp.text)

# Grounding 메타데이터 확인
if resp.candidates[0].grounding_metadata:
    print("\n📚 참고 출처:")
    chunks = resp.candidates[0].grounding_metadata.grounding_chunks or []
    for i, c in enumerate(chunks[:5]):
        if c.web:
            print(f"  [{i+1}] {c.web.title} — {c.web.uri[:80]}")

## 4-11. 도전 과제

### 과제 A — 멀티 함수 + Google Search 결합
다음 시나리오를 처리하는 한 번의 자연어 질의를 작성하세요:
- "경부선 전 구간 점검하고, 만약 비가 오는 구간이 있으면 그 구간에 감속 안내 메시지 송출해줘"
- 이걸 처리하려면 `get_traffic_status` (여러 번) + `google_search` (날씨) + `send_alert_to_dms`를 조합해야 합니다.

### 과제 B — 함수 추가
다음 함수 1개를 추가로 정의하고 시나리오를 만드세요:
- `estimate_travel_time(origin_road_id, destination_road_id)` — 두 지점 간 추정 통행시간
- 이 함수가 추가됐을 때 어떤 새로운 시나리오가 가능해지는가?

### 과제 C — 안전 가드레일 강화
시스템 프롬프트에 다음을 추가하면 어떻게 동작이 바뀌는지 확인:
- "DMS 송출은 운영자에게 메시지 초안을 먼저 확인받아라"
- 이걸 어떻게 구현해야 진짜 안전장치가 될까?

## 4-12. 운영 환경 적용 시 체크리스트

이 핸즈온은 모킹 환경이지만, 실제 ITS 시스템에 적용하려면 다음을 챙겨야 합니다.

| 항목 | 권장 사항 |
|---|---|
| **인프라** | Vertex AI (asia-northeast3) — 데이터 거버넌스, 한국 리전 |
| **인증** | API 키가 아니라 GCP 서비스 계정 + IAM |
| **감사 로그** | 모든 함수 호출 + 입력 + 출력 + 실행자 기록 (공공 사업 필수) |
| **권한 분리** | 조회 함수와 액션 함수의 권한 IAM에서 분리 (위험 함수는 사람 컨펌) |
| **Rate Limit** | DMS 송출 같은 외부 영향 함수는 클라이언트단에서 throttling |
| **롤백** | dispatch/송출 함수는 멱등성(idempotency) 보장 또는 취소 함수 함께 제공 |
| **휴먼 인 더 루프** | 위험도 높은 액션은 모델이 plan만 제시하고 사람이 승인 |

## 4-13. 정리

- ✅ Function Calling 기본 흐름 이해 (모델은 호출 명세만 반환, 실행은 클라이언트)
- ✅ ITS 함수 4종을 tool로 등록
- ✅ 멀티스텝 에이전트 루프 구현
- ✅ Built-in Search Grounding 결합
- ✅ 운영 환경 적용 체크리스트

**핵심 교훈**: Function Calling은 LLM을 "텍스트 생성기"에서 **운영 시스템과 연동되는 에이전트**로 바꾸는 핵심 기능입니다. 다만 **위험한 함수일수록 안전장치가 필수**입니다.

---

**전체 핸즈온 종료** 🎉

5개 노트북에서 다룬 패턴을 정리하면:
1. **00**: 환경 셋업 + 모델 라인업 + thinking_level
2. **01**: 모델·thinking 매트릭스로 비용·성능 정량화
3. **02**: 멀티모달 입력 + Pydantic 스키마 강제 + bounding box
4. **03**: Long Context + Caching으로 대용량 데이터 처리
5. **04**: Function Calling으로 운영 시스템 연동 에이전트

이 5개 패턴이 ITS 영역에서 마주칠 거의 모든 LLM 워크로드를 커버합니다.